# LightGBM Stage 2 — Hyperparameter Grid Search

**Run this BEFORE `lgbm_train.ipynb`.**

Grid search over parameters from **Tyralis et al. (2023)**  
*"Merging satellite and gauge-measured precipitation using LightGBM with an emphasis on extreme quantiles"*  
IEEE JSTARS 16:6969–6979. https://doi.org/10.1109/JSTARS.2023.3297013

Uses last **5 years** of data and an independent 80/20 station split.

Outputs `best_params.json` to S3. `lgbm_train.ipynb` loads it automatically.

**Workflow:**
```
lgbm_hpo.ipynb   →  lgbm/hparam/best_params.json  (S3)
                              ↓
lgbm_train.ipynb  reads best_params.json → trains 5-fold CV → S3
                              ↓
lgbm_eval.ipynb   evaluates trained models
```

## 0. Imports

In [ ]:
import sys, os, gc, json, warnings, itertools, tempfile, shutil, textwrap, subprocess
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, as_completed, ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import properscoring as ps
import boto3

warnings.filterwarnings('ignore')

ROOT = Path('../..')
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'Working directory: {Path.cwd()}')
print(f'LightGBM: {lgb.__version__}')

## 1. Constants

Grid search space from Tyralis et al. (2023) Table 2.  
Constraint: `num_leaves <= 2^max_depth` (as in paper).

> Set `DEVICE = 'cuda'` on Vast.ai GPU, `'cpu'` locally.

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42

# ── Data subset: last 5 years ─────────────────────────────────────────────────
HPO_DATE_START = '2019-01-01'

# ── Train/Val split (independent of lgbm_train fold_assignment) ───────────────
VAL_FRAC = 0.20

# ── Grid search: Tyralis et al. (2023) Table 2 ────────────────────────────────
GRID_MAX_DEPTH         = [6, 8, 10]
GRID_NUM_LEAVES        = [20, 40, 60, 80, 100, 200, 500]
GRID_LEARNING_RATE     = [0.02, 0.05, 0.1]
GRID_MIN_CHILD_SAMPLES = [20, 100, 200, 500, 1000]
HPO_ROUNDS             = 400   # num_iterations fixed per Tyralis et al.
EARLY_STOP             = 20    # early_stopping_round per Tyralis et al.

# ── Parallel quantile workers ─────────────────────────────────────────────────
# Each worker trains one quantile model simultaneously on the same GPU.
# GPU handles concurrent CUDA streams fine. Tune down if you hit OOM.
#   A100 40 GB  → 11  (all quantiles at once)
#   RTX 3090    →  4–6
#   RTX 4090    →  6–8
N_QUANTILE_WORKERS = 6

# ── Quantile levels for CRPS objective ────────────────────────────────────────
QUANTILES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = 'cuda'   # 'cuda' on Vast.ai  |  'cpu' locally

# ── Geo-features (must match lgbm_train.ipynb) ────────────────────────────────
K_GEO         = 15
SVD_QUANTILES = np.arange(0.0, 1.05, 0.05)
SVD_COLS      = [f'svd_{i:02d}' for i in range(len(SVD_QUANTILES))]

# ── Paths ─────────────────────────────────────────────────────────────────────
HPO_DIR = Path('outputs/lgbm/hparam')
HPO_DIR.mkdir(parents=True, exist_ok=True)

FEAT_CACHE = HPO_DIR / 'hpo_features.parquet'
PARAMS_OUT = HPO_DIR / 'best_params.json'
GRID_OUT   = HPO_DIR / 'grid_results.csv'

# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET = 'thesis-data-ismaktam'
S3_HPO    = 'lgbm/hparam'

FORCE_RECOMPUTE = False

# ── Compute valid combos (num_leaves <= 2^max_depth, per Tyralis et al.) ──────
valid_combos = [
    (d, l, lr, m)
    for d, l, lr, m in itertools.product(
        GRID_MAX_DEPTH, GRID_NUM_LEAVES,
        GRID_LEARNING_RATE, GRID_MIN_CHILD_SAMPLES
    )
    if l <= 2 ** d
]
n_models = len(valid_combos) * len(QUANTILES)
speedup  = min(N_QUANTILE_WORKERS, len(QUANTILES))
print(f'Grid: {len(valid_combos)} combos × {len(QUANTILES)} quantiles = {n_models} models')
print(f'Parallel workers: {N_QUANTILE_WORKERS}  (≈{speedup}x faster than sequential)')
print(f'HPO period: {HPO_DATE_START} → cfg.date_end  |  DEVICE={DEVICE}')

## 2. S3 Helpers

In [ ]:
s3 = boto3.client('s3')


def s3_upload(local_path: Path, s3_key: str) -> None:
    try:
        s3.upload_file(str(local_path), S3_BUCKET, s3_key)
        print(f'  ↑ s3://{S3_BUCKET}/{s3_key}')
    except Exception as e:
        print(f'  S3 upload skipped: {e.__class__.__name__}: {e}')


def s3_download(s3_key: str, local_path: Path) -> bool:
    try:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(S3_BUCKET, s3_key, str(local_path))
        print(f'  ↓ s3://{S3_BUCKET}/{s3_key}')
        return True
    except Exception:
        return False


print('S3 helpers ready')

## 3. Load Station Data

In [ ]:
from thesis.config import Config
from thesis.data.registry import DataRegistry
from thesis.transforms import ProjectionTransform, IndicatorTransform
from thesis.transforms.pipeline import TransformPipeline
from thesis.scripts.run_grk_kfold_cv import SOIL_VARS
from thesis.models.grk.features import compute_day_geo_features

cfg      = Config()
registry = DataRegistry.from_config(cfg)

HPO_DATE_END = cfg.date_end
print(f'HPO period: {HPO_DATE_START} → {HPO_DATE_END}')
print('Loading station data...')

all_raw  = registry.stations.load(HPO_DATE_START, HPO_DATE_END)
pipeline = TransformPipeline([
    ProjectionTransform(target_crs=cfg.study_area.target_crs),
    IndicatorTransform(threshold_mm=cfg.wet_day_threshold_mm),
])
all_proc = pipeline.fit_transform(all_raw)
df_wet   = all_proc[all_proc['rain_indicator'] == 1].copy()

station_meta = (
    all_proc
    .drop_duplicates('station_id')[['station_id', 'x_proj', 'y_proj', 'elevation_m']]
    .reset_index(drop=True)
)

print(f'Wet records:  {len(df_wet):,}')
print(f'Stations:     {len(station_meta):,}')
print(f'Unique dates: {df_wet["date"].nunique():,}')

## 4. SoilGrids

In [ ]:
soil_rows = {'station_id': station_meta['station_id'].values}
for var, src in registry.soilgrids.items():
    if var in SOIL_VARS:
        soil_rows[var] = src.sample_at_projected(
            station_meta['x_proj'].values,
            station_meta['y_proj'].values,
        )

soil_static    = pd.DataFrame(soil_rows).set_index('station_id')
available_soil = [v for v in SOIL_VARS if v in soil_static.columns]
for v in available_soil:
    soil_static[v] = soil_static[v].fillna(float(soil_static[v].median()))

FEATURE_COLS = (
    ['x_proj', 'y_proj', 'elevation_m']
    + ['idw', 'gos']
    + SVD_COLS
    + available_soil
)
TARGET_COL = 'precip_mm'

print(f'SoilGrids: {available_soil}')
print(f'Features:  {len(FEATURE_COLS)}')

## 5. Train / Val Station Split

Random 80/20 split on stations — independent of `fold_assignment.parquet`.

In [ ]:
rng        = np.random.default_rng(RANDOM_SEED)
all_sids   = station_meta['station_id'].values
n_val      = int(len(all_sids) * VAL_FRAC)
val_sids   = set(rng.choice(all_sids, size=n_val, replace=False))
train_sids = set(all_sids) - val_sids

print(f'Train stations: {len(train_sids):,}')
print(f'Val stations:   {len(val_sids):,}')
print(f'Val fraction:   {len(val_sids) / len(all_sids):.2f}')

## 6. Geo-Feature Computation

Features are cached locally and on S3.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from joblib import Parallel, delayed

N_JOBS     = os.cpu_count()
BATCH_SIZE = 2000
FEAT_S3    = f'{S3_HPO}/hpo_features.parquet'


def _hpo_day_features(date, grp):
    grp        = grp.reset_index(drop=True)
    xy         = grp[['x_proj', 'y_proj']].values
    z          = grp['precip_mm'].values
    sids       = grp['station_id'].values
    train_mask = np.ones(len(sids), dtype=bool)
    return compute_day_geo_features(date, xy, z, sids, train_mask, K_GEO, SVD_QUANTILES)


if FEAT_CACHE.exists() and not FORCE_RECOMPUTE:
    print(f'Features: local cache ({FEAT_CACHE.name})')
elif not FORCE_RECOMPUTE and s3_download(FEAT_S3, FEAT_CACHE):
    print('Features: downloaded from S3')
else:
    print(f'Features: computing (n_jobs={N_JOBS}, {df_wet["date"].nunique():,} dates)...')
    groups    = list(df_wet.groupby('date'))
    n_batches = (len(groups) + BATCH_SIZE - 1) // BATCH_SIZE
    writer    = None
    total     = 0

    for b in range(n_batches):
        batch   = groups[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=0)(
            delayed(_hpo_day_features)(date, grp) for date, grp in batch
        )
        records = [r for day_recs in results for r in day_recs]
        del results; gc.collect()

        if not records:
            continue
        df_b  = pd.DataFrame(records)
        del records; gc.collect()
        table = pa.Table.from_pandas(df_b, preserve_index=False)
        del df_b; gc.collect()

        if writer is None:
            writer = pq.ParquetWriter(str(FEAT_CACHE), table.schema)
        writer.write_table(table)
        total += len(table)
        del table; gc.collect()
        print(f'  Batch {b+1}/{n_batches}  ({total:,} rows)', end='\r')

    if writer:
        writer.close()
    print(f'  Done — {total:,} rows → {FEAT_CACHE.name}' + ' ' * 20)
    s3_upload(FEAT_CACHE, FEAT_S3)

print(f'Cache size: {FEAT_CACHE.stat().st_size / 1e9:.2f} GB')

## 7. Build Train / Val Arrays

In [ ]:
print('Loading features and building arrays...')

df_geo = pd.read_parquet(FEAT_CACHE)
df_all = (
    df_wet[['station_id', 'date', 'precip_mm', 'x_proj', 'y_proj', 'elevation_m']]
    .merge(df_geo, on=['station_id', 'date'], how='inner')
    .merge(soil_static[available_soil].reset_index(), on='station_id', how='left')
)
for v in available_soil:
    df_all[v] = df_all[v].fillna(df_all[v].median())

del df_geo; gc.collect()

tr_mask = df_all['station_id'].isin(train_sids)
va_mask = df_all['station_id'].isin(val_sids)

X_tr = df_all.loc[tr_mask, FEATURE_COLS].values.astype(np.float32)
y_tr = df_all.loc[tr_mask, TARGET_COL].values.astype(np.float32)
X_va = df_all.loc[va_mask, FEATURE_COLS].values.astype(np.float32)
y_va = df_all.loc[va_mask, TARGET_COL].values.astype(np.float32)

del df_all; gc.collect()

print(f'X_tr: {X_tr.shape}  NaN: {np.isnan(X_tr).sum()}')
print(f'X_va: {X_va.shape}  NaN: {np.isnan(X_va).sum()}')
print(f'y_tr: mean={y_tr.mean():.2f}  max={y_tr.max():.1f} mm')
print(f'y_va: mean={y_va.mean():.2f}  max={y_va.max():.1f} mm')

## 8. Grid Search — Tyralis et al. (2023) Table 2\n\nParameters follow Table 2 from Tyralis et al. (2023):\n- `max_depth` ∈ {6, 8, 10}\n- `num_leaves` ∈ {20, 40, 60, 80, 100, 200, 500}, constraint `num_leaves ≤ 2^max_depth`\n- `learning_rate` ∈ {0.02, 0.05, 0.1}\n- `min_child_samples` ∈ {20, 100, 200, 500, 1000}\n- `num_iterations` = 400 (fixed), `early_stopping_round` = 20\n\nAll other params (`reg_alpha`, `reg_lambda`, `feature_fraction`, `bagging_fraction`) stay at LightGBM defaults, as in Tyralis et al.\n\n**Parallelism:** `N_QUANTILE_WORKERS` quantile models train simultaneously on the GPU via `ThreadPoolExecutor`. Each thread creates its own `lgb.Dataset` — LightGBM's CUDA backend handles concurrent streams safely. Tune `N_QUANTILE_WORKERS` down if you hit OOM."

In [ ]:
# ── Detect available GPUs ─────────────────────────────────────────────────────
try:
    _smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, check=True,
    )
    N_GPUS = len([l for l in _smi.stdout.strip().split('\n') if l.strip()])
except Exception:
    N_GPUS = 1

print(f'GPUs detected: {N_GPUS}')
print(f'Distributing {len(valid_combos)} combos across {N_GPUS} GPU(s)  '
      f'(≈{len(valid_combos) // N_GPUS} combos/GPU, '
      f'{N_QUANTILE_WORKERS} quantile workers/GPU)')

# ── Write GPU worker as importable module (required for ProcessPoolExecutor spawn) ──
WORKER_PY = HPO_DIR / '_lgbm_gpu_worker.py'
WORKER_PY.write_text(textwrap.dedent("""\
    import gc
    import numpy as np
    import lightgbm as lgb
    import properscoring as ps
    from concurrent.futures import ThreadPoolExecutor, as_completed


    def _train_one_quantile(alpha, base_params, X_tr, y_tr, X_va, y_va, hpo_rounds, early_stop):
        dtrain  = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
        dval    = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)
        booster = lgb.train(
            {**base_params, 'alpha': alpha}, dtrain,
            num_boost_round=hpo_rounds, valid_sets=[dval],
            callbacks=[lgb.early_stopping(early_stop, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        preds = np.clip(booster.predict(X_va), 0, None).astype(np.float32)
        del booster, dtrain, dval
        gc.collect()
        return preds


    def run_gpu_batch(args):
        \"\"\"Worker: trains all combos assigned to one GPU, returns list of result dicts.\"\"\"
        gpu_id, combo_batch, data_dir, quantiles, hpo_rounds, early_stop, n_q_workers, random_seed = args

        # Load data fresh in this process (no inherited CUDA context from fork)
        X_tr = np.load(f'{data_dir}/X_tr.npy')
        y_tr = np.load(f'{data_dir}/y_tr.npy')
        X_va = np.load(f'{data_dir}/X_va.npy')
        y_va = np.load(f'{data_dir}/y_va.npy')

        results = []
        for max_depth, num_leaves, lr, min_child in combo_batch:
            base_params = dict(
                objective         = 'quantile',
                metric            = 'quantile',
                max_depth         = max_depth,
                num_leaves        = num_leaves,
                learning_rate     = lr,
                min_child_samples = min_child,
                device_type       = 'cuda',
                gpu_device_id     = gpu_id,
                num_threads       = 4,
                seed              = random_seed,
                verbose           = -1,
            )
            q_preds = np.zeros((len(X_va), len(quantiles)), dtype=np.float32)
            with ThreadPoolExecutor(max_workers=n_q_workers) as pool:
                futures = {
                    pool.submit(
                        _train_one_quantile, alpha, base_params,
                        X_tr, y_tr, X_va, y_va, hpo_rounds, early_stop,
                    ): qi
                    for qi, alpha in enumerate(quantiles)
                }
                for fut in as_completed(futures):
                    q_preds[:, futures[fut]] = fut.result()

            q_preds = np.sort(q_preds, axis=1)
            crps    = float(ps.crps_ensemble(y_va, q_preds).mean())
            mae     = float(np.abs(y_va - q_preds[:, quantiles.index(0.50)]).mean())

            results.append(dict(
                max_depth=max_depth, num_leaves=num_leaves,
                learning_rate=lr, min_child_samples=min_child,
                crps=crps, mae=mae,
            ))
            print(
                f'GPU{gpu_id} | d={max_depth} l={num_leaves:3d} lr={lr:.2f} '
                f'mc={min_child:4d}  CRPS={crps:.4f}',
                flush=True,
            )
        return results
"""))

# Make the worker importable from this notebook's process
if str(HPO_DIR) not in sys.path:
    sys.path.insert(0, str(HPO_DIR))

from _lgbm_gpu_worker import run_gpu_batch  # noqa — import registers it for spawn

# ── Save arrays to a temp dir (each subprocess loads independently) ────────────
tmp_dir = tempfile.mkdtemp(prefix='lgbm_hpo_')
np.save(f'{tmp_dir}/X_tr.npy', X_tr)
np.save(f'{tmp_dir}/y_tr.npy', y_tr)
np.save(f'{tmp_dir}/X_va.npy', X_va)
np.save(f'{tmp_dir}/y_va.npy', y_va)
_tmp_mb = sum(os.path.getsize(f'{tmp_dir}/{f}') for f in os.listdir(tmp_dir)) / 1e6
print(f'Arrays saved to {tmp_dir}  ({_tmp_mb:.0f} MB)')

# ── Distribute combos round-robin across GPUs ─────────────────────────────────
batches = [valid_combos[i::N_GPUS] for i in range(N_GPUS)]
for i, b in enumerate(batches):
    print(f'  GPU{i}: {len(b)} combos')

worker_args = [
    (gpu_id, batches[gpu_id], tmp_dir, QUANTILES,
     HPO_ROUNDS, EARLY_STOP, N_QUANTILE_WORKERS, RANDOM_SEED)
    for gpu_id in range(N_GPUS)
]

# ── Launch grid search ────────────────────────────────────────────────────────
print(f'\nStarting grid search  (HPO_ROUNDS={HPO_ROUNDS}  EARLY_STOP={EARLY_STOP})\n')

if N_GPUS == 1:
    # Single GPU — run directly, no subprocess overhead
    grid_results = run_gpu_batch(worker_args[0])
else:
    # Multi-GPU — one subprocess per GPU, each with its own CUDA context
    ctx = mp.get_context('spawn')
    with ProcessPoolExecutor(max_workers=N_GPUS, mp_context=ctx) as pool:
        futures = {pool.submit(run_gpu_batch, args): args[0] for args in worker_args}
        grid_results = []
        for fut in as_completed(futures):
            gpu_id       = futures[fut]
            batch_res    = fut.result()
            grid_results.extend(batch_res)
            best_so_far  = min(r['crps'] for r in grid_results)
            print(f'GPU{gpu_id} finished: {len(batch_res)} combos  |  best CRPS so far: {best_so_far:.4f}')

# ── Cleanup temp dir ──────────────────────────────────────────────────────────
shutil.rmtree(tmp_dir, ignore_errors=True)

print(f'\nGrid search complete: {len(grid_results)} / {len(valid_combos)} combos')

## 9. Results

In [ ]:
df_results = pd.DataFrame(grid_results).sort_values('crps').reset_index(drop=True)

print('=' * 70)
print('TOP 15 CONFIGURATIONS')
print('=' * 70)
print(df_results.head(15).to_string(index=True))

best_row = df_results.iloc[0]
print(f'\n{"=" * 70}')
print(f'BEST  CRPS={best_row["crps"]:.4f}  MAE={best_row["mae"]:.4f}')
print(f'  max_depth={int(best_row["max_depth"])}  num_leaves={int(best_row["num_leaves"])}  '
      f'learning_rate={best_row["learning_rate"]}  min_child_samples={int(best_row["min_child_samples"])}')
print(f'{"=" * 70}')

# ── Marginal means: mean CRPS when each param is fixed at a value ─────────────
print('\n── Marginal analysis (mean CRPS per parameter value) ──')
for param, vals in [
    ('learning_rate',     GRID_LEARNING_RATE),
    ('num_leaves',        GRID_NUM_LEAVES),
    ('max_depth',         GRID_MAX_DEPTH),
    ('min_child_samples', GRID_MIN_CHILD_SAMPLES),
]:
    row = '  '.join(
        f'{v}: {df_results[df_results[param] == v]["crps"].mean():.4f}'
        for v in vals if v in df_results[param].values
    )
    print(f'  {param:<22s}  {row}')

# ── CRPS range summary ────────────────────────────────────────────────────────
print(f'\n── CRPS range: min={df_results["crps"].min():.4f}  '
      f'median={df_results["crps"].median():.4f}  '
      f'max={df_results["crps"].max():.4f}')

In [ ]:
# ── Figure 1: Overview ────────────────────────────────────────────────────────
fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle('Grid Search Overview', fontsize=14, fontweight='bold')

# 1a: CRPS distribution
ax = axes[0, 0]
ax.hist(df_results['crps'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(best_row['crps'], color='red', lw=2, label=f'Best={best_row["crps"]:.4f}')
ax.axvline(df_results['crps'].median(), color='orange', lw=1.5, linestyle='--',
           label=f'Median={df_results["crps"].median():.4f}')
ax.set_xlabel('CRPS')
ax.set_ylabel('Count')
ax.set_title('CRPS distribution across all 240 configs')
ax.legend(fontsize=8)

# 1b: CRPS vs num_leaves, coloured by learning_rate
ax = axes[0, 1]
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for c, lr in zip(colors, GRID_LEARNING_RATE):
    sub = df_results[df_results['learning_rate'] == lr]
    # show best CRPS per num_leaves for this lr
    grp = sub.groupby('num_leaves')['crps'].min().reset_index()
    ax.plot(grp['num_leaves'], grp['crps'], marker='o', ms=5, label=f'lr={lr}', color=c)
ax.set_xlabel('num_leaves')
ax.set_ylabel('Best CRPS (min over other params)')
ax.set_title('Best CRPS per num_leaves by learning_rate')
ax.legend(fontsize=8)

# 1c: Box plots by learning_rate
ax = axes[1, 0]
data_lr  = [df_results[df_results['learning_rate'] == lr]['crps'].values for lr in GRID_LEARNING_RATE]
bp = ax.boxplot(data_lr, labels=[str(lr) for lr in GRID_LEARNING_RATE], patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_xlabel('learning_rate')
ax.set_ylabel('CRPS')
ax.set_title('CRPS distribution by learning_rate')

# 1d: Box plots by max_depth
ax = axes[1, 1]
colors_d = ['#984ea3', '#ff7f00', '#a65628']
data_d   = [df_results[df_results['max_depth'] == d]['crps'].values for d in GRID_MAX_DEPTH]
bp = ax.boxplot(data_d, labels=[str(d) for d in GRID_MAX_DEPTH], patch_artist=True)
for patch, c in zip(bp['boxes'], colors_d):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_xlabel('max_depth')
ax.set_ylabel('CRPS')
ax.set_title('CRPS distribution by max_depth')

fig1.tight_layout()
fig1.savefig(HPO_DIR / 'hpo_fig1_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hpo_fig1_overview.png')

In [ ]:
# ── Figure 2: Heatmaps — num_leaves × learning_rate for each max_depth ────────
fig2, axes = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle('CRPS Heatmap: num_leaves × learning_rate  (min over min_child_samples)',
              fontsize=13, fontweight='bold')

import matplotlib.colors as mcolors

vmin = df_results['crps'].min()
vmax = df_results['crps'].quantile(0.75)   # cap colourscale at 75th pct for contrast

for ax, depth in zip(axes, GRID_MAX_DEPTH):
    sub   = df_results[df_results['max_depth'] == depth]
    valid_leaves = sorted([l for l in GRID_NUM_LEAVES if l <= 2 ** depth])
    pivot = (
        sub.groupby(['num_leaves', 'learning_rate'])['crps']
        .min()
        .unstack('learning_rate')
        .reindex(valid_leaves)
    )
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r',
                   vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(i) for i in pivot.index], fontsize=9)
    ax.set_xlabel('learning_rate')
    ax.set_ylabel('num_leaves')
    ax.set_title(f'max_depth={depth}')
    # annotate cells
    for row_i in range(pivot.shape[0]):
        for col_j in range(pivot.shape[1]):
            val = pivot.values[row_i, col_j]
            if not np.isnan(val):
                ax.text(col_j, row_i, f'{val:.3f}', ha='center', va='center',
                        fontsize=7, color='black')
    plt.colorbar(im, ax=ax, shrink=0.85, label='CRPS')

fig2.tight_layout()
fig2.savefig(HPO_DIR / 'hpo_fig2_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hpo_fig2_heatmaps.png')

In [ ]:
# ── Figure 3: Marginal effects — mean & min CRPS per parameter value ──────────
fig3, axes = plt.subplots(2, 2, figsize=(14, 10))
fig3.suptitle('Marginal Effects: CRPS per Parameter Value', fontsize=13, fontweight='bold')

param_specs = [
    ('learning_rate',     GRID_LEARNING_RATE,     axes[0, 0]),
    ('num_leaves',        GRID_NUM_LEAVES,         axes[0, 1]),
    ('max_depth',         GRID_MAX_DEPTH,          axes[1, 0]),
    ('min_child_samples', GRID_MIN_CHILD_SAMPLES,  axes[1, 1]),
]

for param, vals, ax in param_specs:
    present = [v for v in vals if v in df_results[param].values]
    means   = [df_results[df_results[param] == v]['crps'].mean() for v in present]
    mins    = [df_results[df_results[param] == v]['crps'].min()  for v in present]
    x       = range(len(present))

    ax.bar(x, means, color='steelblue', alpha=0.6, label='Mean CRPS')
    ax.plot(x, mins, 'r-o', ms=6, lw=2, label='Best CRPS')
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in present], fontsize=9)
    ax.set_xlabel(param)
    ax.set_ylabel('CRPS')
    ax.set_title(f'Effect of {param}')
    ax.legend(fontsize=8)

    # label the best bar
    best_idx = int(np.argmin(mins))
    ax.annotate(f'{mins[best_idx]:.4f}',
                xy=(best_idx, mins[best_idx]),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='red')

fig3.tight_layout()
fig3.savefig(HPO_DIR / 'hpo_fig3_marginals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hpo_fig3_marginals.png')

In [ ]:
# ── Figure 4: Interaction — num_leaves × min_child_samples per learning_rate ──
fig4, axes = plt.subplots(1, 3, figsize=(18, 5))
fig4.suptitle('Interaction: num_leaves × min_child_samples\n(each panel = one learning_rate, '
              'lines = min_child_samples, y = best CRPS over max_depth)',
              fontsize=11, fontweight='bold')

cmap_mc = plt.cm.get_cmap('tab10', len(GRID_MIN_CHILD_SAMPLES))

for ax, lr in zip(axes, GRID_LEARNING_RATE):
    sub = df_results[df_results['learning_rate'] == lr]
    for ci, mc in enumerate(GRID_MIN_CHILD_SAMPLES):
        sub_mc = sub[sub['min_child_samples'] == mc]
        if sub_mc.empty:
            continue
        grp = sub_mc.groupby('num_leaves')['crps'].min().reset_index()
        ax.plot(grp['num_leaves'], grp['crps'], marker='o', ms=5,
                color=cmap_mc(ci), label=f'mc={mc}')
    ax.set_xlabel('num_leaves')
    ax.set_ylabel('Best CRPS (min over max_depth)')
    ax.set_title(f'learning_rate = {lr}')
    ax.legend(fontsize=7, title='min_child_samples')

fig4.tight_layout()
fig4.savefig(HPO_DIR / 'hpo_fig4_interactions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hpo_fig4_interactions.png')

## 10. Save Best Params + Upload to S3

In [ ]:
# ── best_params.json ──────────────────────────────────────────────────────────
best_params = {
    'max_depth':          int(best_row['max_depth']),
    'num_leaves':         int(best_row['num_leaves']),
    'learning_rate':      float(best_row['learning_rate']),
    'min_child_samples':  int(best_row['min_child_samples']),
    'crps_val':           float(best_row['crps']),
    'mae_val':            float(best_row['mae']),
    'hpo_period':         f'{HPO_DATE_START} → {HPO_DATE_END}',
    'source':             'grid_search_tyralis2023',
    'n_combos':           len(df_results),
    'n_quantile_workers': N_QUANTILE_WORKERS,
}

with open(PARAMS_OUT, 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'Saved: {PARAMS_OUT}')
print(json.dumps(best_params, indent=2))

# ── grid_results.csv — full table for offline analysis ────────────────────────
df_save = df_results.copy()
df_save.insert(0, 'rank', range(1, len(df_save) + 1))
df_save['crps_gap']     = df_save['crps'] - df_save['crps'].min()
df_save['crps_gap_pct'] = df_save['crps_gap'] / df_save['crps'].min() * 100

df_save.to_csv(GRID_OUT, index=False, float_format='%.6f')
print(f'Saved: {GRID_OUT}  ({len(df_save)} rows, {df_save.shape[1]} cols)')

# ── summary.json — marginal stats for quick reference ─────────────────────────
SUMMARY_OUT = HPO_DIR / 'grid_summary.json'
summary = {
    'best': best_params,
    'crps': {
        'min':    float(df_results['crps'].min()),
        'median': float(df_results['crps'].median()),
        'max':    float(df_results['crps'].max()),
        'std':    float(df_results['crps'].std()),
    },
    'marginal_best_crps': {
        param: {
            str(v): float(df_results[df_results[param] == v]['crps'].min())
            for v in vals if v in df_results[param].values
        }
        for param, vals in [
            ('learning_rate',     GRID_LEARNING_RATE),
            ('num_leaves',        GRID_NUM_LEAVES),
            ('max_depth',         GRID_MAX_DEPTH),
            ('min_child_samples', GRID_MIN_CHILD_SAMPLES),
        ]
    },
    'marginal_mean_crps': {
        param: {
            str(v): float(df_results[df_results[param] == v]['crps'].mean())
            for v in vals if v in df_results[param].values
        }
        for param, vals in [
            ('learning_rate',     GRID_LEARNING_RATE),
            ('num_leaves',        GRID_NUM_LEAVES),
            ('max_depth',         GRID_MAX_DEPTH),
            ('min_child_samples', GRID_MIN_CHILD_SAMPLES),
        ]
    },
    'top5': df_save.head(5)[
        ['rank', 'max_depth', 'num_leaves', 'learning_rate', 'min_child_samples', 'crps', 'mae']
    ].to_dict(orient='records'),
}

with open(SUMMARY_OUT, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved: {SUMMARY_OUT}')

# ── Upload everything to S3 ────────────────────────────────────────────────────
uploads = [
    (PARAMS_OUT,                             f'{S3_HPO}/best_params.json'),
    (GRID_OUT,                               f'{S3_HPO}/grid_results.csv'),
    (SUMMARY_OUT,                            f'{S3_HPO}/grid_summary.json'),
    (HPO_DIR / 'hpo_fig1_overview.png',      f'{S3_HPO}/hpo_fig1_overview.png'),
    (HPO_DIR / 'hpo_fig2_heatmaps.png',      f'{S3_HPO}/hpo_fig2_heatmaps.png'),
    (HPO_DIR / 'hpo_fig3_marginals.png',     f'{S3_HPO}/hpo_fig3_marginals.png'),
    (HPO_DIR / 'hpo_fig4_interactions.png',  f'{S3_HPO}/hpo_fig4_interactions.png'),
]

print('\nUploading to S3...')
for local, key in uploads:
    if Path(local).exists():
        s3_upload(Path(local), key)
    else:
        print(f'  SKIP (not found): {local}')

print('\nDone. S3 layout:')
for _, key in uploads:
    print(f'  s3://{S3_BUCKET}/{key}')

## 11. Next Step

To wire these HPO results into `lgbm_train.ipynb`, add the following cell right after the constants block:

```python
hpo_path = Path('outputs/lgbm/hparam/best_params.json')
if hpo_path.exists():
    with open(hpo_path) as f:
        _hpo = json.load(f)
    _hpo_keys = ('learning_rate', 'num_leaves', 'max_depth', 'min_child_samples')
    LGB_BASE.update({k: v for k, v in _hpo.items() if k in _hpo_keys})
    print(f'LGB_BASE updated from HPO: CRPS_val={_hpo.get("crps_val", "?"):.4f}')
else:
    print('No HPO results found — using default LGB_BASE')
```